In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, recall_score, precision_score, roc_curve, auc, roc_auc_score

In [ ]:
data = pd.read_excel('/content/sample_data/TelecomCustomerChurn.xlsx')
data.shape

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data['Total Charges'] = pd.to_numeric(data['Total Charges'], errors='coerce')
original_rows = len(data)
data = data.dropna(subset=['Total Charges'])
print(f"Total dropped rows with NaNs: {original_rows - len(data)}")

In [ ]:
data.columns

In [ ]:
target_feature = 'Churn'
numeric_features = ['MonthlyCharges', 'TotalCharges', 'tenure']
categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
      'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod']

In [ ]:
numeric_features = [
 'Age', 'Number of Dependents', 'Number of Referrals', 'Tenure in Months',
 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download',
 'Monthly Charge', 'Total Charges', 'Total Refunds',
 'Total Extra Data Charges', 'Total Long Distance Charges', 'Total Revenue'
]

categorical_features = [
 'Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines',
 'Internet Service', 'Internet Type', 'Online Security', 'Online Backup',
 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV',
 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract',
 'Paperless Billing', 'Payment Method'
]

target_feature = 'Customer Status'

X = data[numeric_features + categorical_features]
y = data[target_feature]

In [ ]:
sample_data = pd.DataFrame({
    "Age": np.random.randint(18, 80, size=50),
    "Number of Dependents": np.random.randint(0, 5, size=50),
    "Number of Referrals": np.random.randint(0, 10, size=50),
    "Tenure in Months": np.random.randint(0, 72, size=50),
    "Avg Monthly Long Distance Charges": np.random.uniform(0, 50, size=50),
    "Avg Monthly GB Download": np.random.uniform(0, 100, size=50),
    "Monthly Charge": np.random.uniform(20, 120, size=50),
    "Total Charges": np.random.uniform(100, 5000, size=50),
    "Total Refunds": np.random.uniform(0, 100, size=50),
    "Total Extra Data Charges": np.random.uniform(0, 200, size=50),
    "Total Long Distance Charges": np.random.uniform(0, 300, size=50),
    "Total Revenue": np.random.uniform(500, 10000, size=50),
    "Gender": np.random.choice(["Male", "Female"], size=50),
    "Married": np.random.choice(["Yes", "No"], size=50),
    "Offer": np.random.choice(["None", "Offer A", "Offer B"], size=50),
    "Phone Service": np.random.choice(["Yes", "No"], size=50),
    "Multiple Lines": np.random.choice(["Yes", "No"], size=50),
    "Internet Service": np.random.choice(["DSL", "Fiber optic", "No"], size=50),
    "Internet Type": np.random.choice(["Cable", "Fiber", "None"], size=50),
    "Online Security": np.random.choice(["Yes", "No"], size=50),
    "Online Backup": np.random.choice(["Yes", "No"], size=50),
    "Device Protection Plan": np.random.choice(["Yes", "No"], size=50),
    "Premium Tech Support": np.random.choice(["Yes", "No"], size=50),
    "Streaming TV": np.random.choice(["Yes", "No"], size=50),
    "Streaming Movies": np.random.choice(["Yes", "No"], size=50),
    "Streaming Music": np.random.choice(["Yes", "No"], size=50),
    "Unlimited Data": np.random.choice(["Yes", "No"], size=50),
    "Contract": np.random.choice(["Month-to-month", "One year", "Two year"], size=50),
    "Paperless Billing": np.random.choice(["Yes", "No"], size=50),
    "Payment Method": np.random.choice([
        "Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"
    ], size=50)
})


In [ ]:
from collections import Counter
Counter(y)
#The data is imbalanced since yes and no are not 50-50

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, stratify=y)
#In train_test_split, the stratify parameter ensures that the proportions of class labels in the training and testing sets are approximately the same as the proportions in the original dataset

In [ ]:
y_train = y_train.map({'Churned': 1, 'Stayed': 0, 'Joined': 0})
y_test = y_test.map({'Churned': 1, 'Stayed': 0, 'Joined': 0})
y = y.map({'Churned': 1, 'Stayed': 0, 'Joined': 0})

In [ ]:
print(f"Original data churn rate: {y.mean():.2f}")
print(f"Training data churn rate: {y_train.mean():.2f}")
print(f"Testing data churn rate: {y_test.mean():.2f}")

In [ ]:
from sklearn.compose import ColumnTransformer
# Numeric Preprocessing Batch

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
# Categorical Preprocessing Batch

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
# Combine both batches with ColumnTransformer

preprocessor = ColumnTransformer(
    transformers = [
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)

    ],
    remainder = 'drop'
)
# Create the final, full-stack pipeline
clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight = 'balanced',
        random_state = 42,

    ))
])

In [ ]:
# Re-initialize y from the original data and re-split for y_train, y_test.
# This is necessary because y, y_train, and y_test currently contain NaN values
# due to previous execution with an incorrect mapping.

# 1. Re-create y from data (assuming 'data' and 'target_feature' are still intact)
y = data[target_feature]

# 2. Re-split data to get fresh y_train and y_test with original string values
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

# 3. Apply the correct mapping to y_train and y_test
y_train = y_train.map({'Churned': 1, 'Stayed': 0, 'Joined': 0})
y_test = y_test.map({'Churned': 1, 'Stayed': 0, 'Joined': 0})

clf_pipeline.fit(X_train, y_train)

In [ ]:
y_predict = clf_pipeline.predict(X_test)
print(y_predict)

In [ ]:
y_proba = clf_pipeline.predict_proba(X_test)[:, 1]
print(y_proba)

In [ ]:
print(recall_score(y_test, y_predict))

In [ ]:
print(confusion_matrix(y_test, y_predict))

In [ ]:
print(precision_score(y_test, y_predict))

In [ ]:
print(accuracy_score(y_test, y_predict))

In [ ]:
print(f1_score(y_test, y_predict))

In [ ]:
print(classification_report(y_test, y_predict))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
print(roc_auc)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators = 200,
        class_weight = 'balanced',
        random_state = 42,

    ))
])

In [ ]:
rf_model.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
print(classification_report(y_test, y_pred_rf))

In [ ]:
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_proba_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)
print(roc_auc_rf)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(fpr_rf, tpr_rf, color='darkorange', lw=2, label=f"Random Forest (AUC = {roc_auc_rf:.2f})")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Random Forest")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

cv_auc = cross_val_score(
    clf_pipeline, X_train, y_train,
    scoring='roc_auc',
    cv=5
)
print("Logistic Regression CV AUC:", cv_auc.mean())

In [ ]:
cv_auc = cross_val_score(
    rf_model, X_train, y_train,
    scoring='roc_auc',
    cv=5
)
print("Random Forest CV AUC:", cv_auc.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 10],
    'model__min_samples_leaf': [1, 5]
}

In [ ]:
grid_rf = GridSearchCV(
    rf_model,
    param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_rf.fit(X_train, y_train)

In [ ]:
print("Best RF Params:", grid_rf.best_params_)
print("Best RF CV AUC:", grid_rf.best_score_)

In [ ]:
best_rf = grid_rf.best_estimator_

y_pred_rf = best_rf.predict(X_test)
y_pred_proba_rf = best_rf.predict_proba(X_test)[:, 1]

fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_pred_proba_rf)
roc_auc_rf = auc(fpr_rf, tpr_rf)
print(roc_auc_rf)
print(accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

In [ ]:
feature_names = best_rf.named_steps['preprocessor'].get_feature_names_out()
importances = best_rf.named_steps['model'].feature_importances_

feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

feat_imp.head(15)

In [ ]:
top_features = feat_imp.head(15)
plt.figure(figsize=(10, 6))
plt.barh(top_features['feature'], top_features['importance'])
plt.gca().invert_yaxis()
plt.title("Top Drivers of Customer Churn")
plt.show()

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)

results = []

for t in thresholds:
    y_pred_custom = (y_pred_proba_rf >= t).astype(int)

    precision = precision_score(y_test, y_pred_custom)
    recall = recall_score(y_test, y_pred_custom)
    f1 = f1_score(y_test, y_pred_custom)

    results.append((t, precision, recall, f1))

results_df = pd.DataFrame(results, columns=["threshold", "precision", "recall", "f1"])
results_df

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(results_df["threshold"], results_df["precision"], label="Precision")
plt.plot(results_df["threshold"], results_df["recall"], label="Recall")
plt.plot(results_df["threshold"], results_df["f1"], label="F1 Score")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.legend()
plt.title("Precision-Recall Tradeoff by Threshold")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
y_pred_final = best_rf.predict(X_test)

cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - Final Model")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(fpr, tpr, label=f"Logistic Regression (AUC={roc_auc:.2f})")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={roc_auc_rf:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Model Comparison ROC Curve")
plt.legend()
plt.show()

In [ ]:
churn_prob = y_pred_proba_rf

risk_df = pd.DataFrame({
    "churn_probability": churn_prob
})

risk_df["risk_level"] = pd.cut(
    risk_df["churn_probability"],
    bins=[0,0.3,0.6,1],
    labels=["Low Risk","Medium Risk","High Risk"]
)

risk_df["risk_level"].value_counts()

In [ ]:
import joblib

joblib.dump(clf_pipeline, "logistic_model.pkl")
joblib.dump(best_rf, "rf_model.pkl")

In [ ]:
import sklearn
print(sklearn.__version__)

In [ ]:
import shap
X_train_transformed = best_rf.named_steps['preprocessor'].transform(X_train)

In [ ]:
feature_names = best_rf.named_steps['preprocessor'].get_feature_names_out()

In [ ]:
rf_classifier = best_rf.named_steps['model']
explainer = shap.TreeExplainer(rf_classifier)
shap_values = explainer.shap_values(X_train_transformed)

In [ ]:
print(type(shap_values))
print(len(shap_values))
print(shap_values[1].shape)
print(X_train_transformed.shape)

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X_train_transformed,
    feature_names=feature_names
)

In [ ]:
shap.summary_plot(
    shap_values[:, :, 1],
    X_train_transformed,
    feature_names=feature_names,
    plot_type="bar"
)

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos

In [ ]:
xgb_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    ))
])

In [ ]:
xgb_model.fit(X_train, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_xgb))
print("ROC AUC:", roc_auc_score(y_test, y_proba_xgb))

In [ ]:
performance_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC AUC": [0.86, 0.85, round(roc_auc_score(y_test, y_proba_xgb), 2)],
    "F1 Score": [0.64, 0.65, round(f1_score(y_test, y_pred_xgb), 2)],
    "Precision": [0.52, 0.56, round(precision_score(y_test, y_pred_xgb), 2)],
    "Recall": [0.84, 0.78, round(recall_score(y_test, y_pred_xgb), 2)]
})
performance_df

In [ ]:
!mkdir models

In [ ]:
import joblib
joblib.dump(clf_pipeline, "models/logistic_model.pkl")
joblib.dump(rf_model, "models/rf_model.pkl")
joblib.dump(xgb_model, "models/xgb_model.pkl")